# full test inference

five checkpointed methods on the full DigiCam test split

In [9]:
from dataclasses import dataclass
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BASE_OUTPUT = ROOT / "outputs" / "full_test_all_methods"
MASKS_DIR = ROOT / "data" / "digicam_masks"
NUM_WORKERS = 4


@dataclass(frozen=True)
class Method:
    name: str
    config: str
    checkpoint: str
    output_name: str
    overrides: tuple[str, ...] = ()


METHODS = [
    Method("LeADMM-20", "inference_leadmm20", "checkpoints/leadmm20.pth", "leadmm20"),
    Method(
        "LeADMM-5 + DRUNet pre",
        "inference_modular_pre",
        "checkpoints/leadmm5_pre_drunet8m.pth",
        "leadmm5_drunet_pre",
        ("model=modular_leadmm5_pre_drunet8m",),
    ),
    Method(
        "LeADMM-5 + DRUNet post",
        "inference_modular_post",
        "checkpoints/leadmm5_post_drunet8m.pth",
        "leadmm5_drunet_post",
        ("model=modular_leadmm5_post_drunet8m",),
    ),
    Method(
        "LeADMM-5 + DRUNet pre+post",
        "inference_modular_prepost",
        "checkpoints/leadmm5_prepost_drunet8m.pth",
        "leadmm5_drunet_prepost",
        ("model=modular_leadmm5_prepost_drunet8m",),
    ),
    Method("FlatNet-lite", "inference_flatnet", "checkpoints/flatnet_lite.pth", "flatnet_lite"),
]

BASE_OUTPUT.mkdir(parents=True, exist_ok=True)

if not all((ROOT / method.checkpoint).is_file() for method in METHODS):
    subprocess.run([sys.executable, "download_checkpoints.py", "--all"], cwd=ROOT, check=True)

if not any(MASKS_DIR.glob("mask_*.npy")):
    subprocess.run([sys.executable, "scripts/download_digicam_masks.py"], cwd=ROOT, check=True)

assert torch.cuda.is_available(), "CUDA GPU is required"


def rel(path: Path) -> str:
    return str(path.relative_to(ROOT))

display(
    pd.DataFrame(
        [
            {
                "method": method.name,
                "config": method.config,
                "checkpoint": method.checkpoint,
                "output_dir": rel(BASE_OUTPUT / method.output_name),
            }
            for method in METHODS
        ]
    )
)
print("device: cuda")

,method,config,checkpoint,output_dir
0,LeADMM-20,inference_leadmm20,checkpoints/leadmm20.pth,outputs/full_test_all_methods/leadmm20
1,LeADMM-5 + DRUNet pre,inference_modular_pre,checkpoints/leadmm5_pre_drunet8m.pth,outputs/full_test_all_methods/leadmm5_drunet_pre
2,LeADMM-5 + DRUNet post,inference_modular_post,checkpoints/leadmm5_post_drunet8m.pth,outputs/full_test_all_methods/leadmm5_drunet_post
3,LeADMM-5 + DRUNet pre+post,inference_modular_prepost,checkpoints/leadmm5_prepost_drunet8m.pth,outputs/full_test_all_methods/leadmm5_drunet_p...
4,FlatNet-lite,inference_flatnet,checkpoints/flatnet_lite.pth,outputs/full_test_all_methods/flatnet_lite


device: cuda


## run inference

In [10]:
def run_nohup(cmd, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            ["nohup", *cmd],
            cwd=ROOT,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
        return_code = process.wait()
    if return_code != 0:
        tail = "".join(log_path.read_text(encoding="utf-8", errors="replace").splitlines(True)[-40:])
        raise RuntimeError(f"failed: {' '.join(cmd)}\n\n{tail}")
    return log_path


def run_inference(method: Method):
    output_dir = BASE_OUTPUT / method.output_name
    cmd = [
        sys.executable,
        "inference.py",
        "-cn",
        method.config,
        *method.overrides,
        f"checkpoint={method.checkpoint}",
        f"output_dir={rel(output_dir)}",
        "device=cuda",
        f"num_workers={NUM_WORKERS}",
    ]
    log_path = run_nohup(cmd, output_dir / "inference.log")
    return {
        "method": method.name,
        "device": "cuda",
        "output_dir": rel(output_dir),
        "log": rel(log_path),
    }


inference_rows = [run_inference(method) for method in METHODS]
display(pd.DataFrame(inference_rows))

,method,device,output_dir,log
0,LeADMM-20,cuda,outputs/full_test_all_methods/leadmm20,outputs/full_test_all_methods/leadmm20/inferen...
1,LeADMM-5 + DRUNet pre,cuda,outputs/full_test_all_methods/leadmm5_drunet_pre,outputs/full_test_all_methods/leadmm5_drunet_p...
2,LeADMM-5 + DRUNet post,cuda,outputs/full_test_all_methods/leadmm5_drunet_post,outputs/full_test_all_methods/leadmm5_drunet_p...
3,LeADMM-5 + DRUNet pre+post,cuda,outputs/full_test_all_methods/leadmm5_drunet_p...,outputs/full_test_all_methods/leadmm5_drunet_p...
4,FlatNet-lite,cuda,outputs/full_test_all_methods/flatnet_lite,outputs/full_test_all_methods/flatnet_lite/inf...


## compute ROI metrics

In [11]:
def run_metrics(method: Method):
    output_dir = BASE_OUTPUT / method.output_name
    metrics_path = output_dir / "metrics.json"
    cmd = [
        sys.executable,
        "calculate_metrics.py",
        f"pred_dir={rel(output_dir / 'recon')}",
        f"gt_dir={rel(output_dir / 'lensed')}",
        f"output_path={rel(metrics_path)}",
        "device=cuda",
    ]
    log_path = run_nohup(cmd, output_dir / "metrics.log")
    return {
        "method": method.name,
        "device": "cuda",
        "metrics": rel(metrics_path),
        "log": rel(log_path),
    }


metric_rows = [run_metrics(method) for method in METHODS]
display(pd.DataFrame(metric_rows))

,method,device,metrics,log
0,LeADMM-20,cuda,outputs/full_test_all_methods/leadmm20/metrics...,outputs/full_test_all_methods/leadmm20/metrics...
1,LeADMM-5 + DRUNet pre,cuda,outputs/full_test_all_methods/leadmm5_drunet_p...,outputs/full_test_all_methods/leadmm5_drunet_p...
2,LeADMM-5 + DRUNet post,cuda,outputs/full_test_all_methods/leadmm5_drunet_p...,outputs/full_test_all_methods/leadmm5_drunet_p...
3,LeADMM-5 + DRUNet pre+post,cuda,outputs/full_test_all_methods/leadmm5_drunet_p...,outputs/full_test_all_methods/leadmm5_drunet_p...
4,FlatNet-lite,cuda,outputs/full_test_all_methods/flatnet_lite/met...,outputs/full_test_all_methods/flatnet_lite/met...


## result

In [12]:
rows = []
for method in METHODS:
    output_dir = BASE_OUTPUT / method.output_name
    metrics_path = output_dir / "metrics.json"
    metrics = json.loads(metrics_path.read_text())
    recon_count = len(list((output_dir / "recon").glob("*.png")))
    gt_count = len(list((output_dir / "lensed").glob("*.png")))
    rows.append(
        {
            "method": method.name,
            "split": "test",
            "recon_images": recon_count,
            "gt_images": gt_count,
            "num_images": metrics["num_images"],
            "psnr": metrics["PSNR"],
            "ssim": metrics["SSIM"],
            "mse": metrics["MSE"],
            "lpips": metrics["LPIPS"],
        }
    )

summary = pd.DataFrame(rows).sort_values("psnr", ascending=False).reset_index(drop=True)
summary.to_csv(BASE_OUTPUT / "summary.csv", index=False)
display(summary)
assert (summary["recon_images"] == summary["gt_images"]).all()
assert (summary["num_images"] == summary["recon_images"]).all()
assert summary["num_images"].min() >= 1500

,method,split,recon_images,gt_images,num_images,psnr,ssim,mse,lpips
0,LeADMM-5 + DRUNet pre+post,test,1500,1500,1500,17.312799,0.444584,0.023172,0.576596
1,LeADMM-5 + DRUNet post,test,1500,1500,1500,16.685324,0.422375,0.026314,0.596452
2,FlatNet-lite,test,1500,1500,1500,16.121000,0.380148,0.029315,0.605604
3,LeADMM-5 + DRUNet pre,test,1500,1500,1500,15.294154,0.310417,0.034777,0.685932
4,LeADMM-20,test,1500,1500,1500,8.152434,0.237773,0.159946,0.765923
